## Finding Dataset

In order to realize the study, datasets representing the moon were searched for, with the NASA considered the best. It offered an easy api to obtain data as a detailed dataset displaying moon diameter distance phase and age in number, which offers much value to the sucessing stages. The NASA dataset divides itself per year, so calling 6 diferent urls were necessary to get the number from 2020 to 2025
Before continuing, the dataset was downloaded to a file for the manual read of the data.

In [33]:
import pandas as pd

url = {
        2025: "https://svs.gsfc.nasa.gov/vis/a000000/a005400/a005415/mooninfo_2025.json",
        2024: "https://svs.gsfc.nasa.gov/vis/a000000/a005100/a005187/mooninfo_2024.json",
        2023: "https://svs.gsfc.nasa.gov/vis/a000000/a005000/a005048/mooninfo_2023.json",
        2022: "https://svs.gsfc.nasa.gov/vis/a000000/a004900/a004955/mooninfo_2022.json",
        2021: "https://svs.gsfc.nasa.gov/vis/a000000/a004800/a004874/mooninfo_2021.json",
        2020: "https://svs.gsfc.nasa.gov/vis/a000000/a004700/a004768/mooninfo_2020.json", 
}

frames = []

for y, u in url.items():
    print(f"Downloading {y}")
    try:
        df_year = pd.read_json(u)
        df_year["year"] = y
        frames.append(df_year)
    except Exception as e:
        print(f"Failed to download year {y}")

df = pd.concat(frames, ignore_index=True)

print("Downloaded Moon phases datasets over 2020", len(df), "records")
print(df.columns)


Downloaded Moon phases datasets over 2020 52610 records
Index(['time', 'phase', 'age', 'diameter', 'distance', 'j2000', 'subsolar',
       'subearth', 'posangle', 'year'],
      dtype='object')


After the retrieval of the raw dataset, the step of cleaning and deciding what columns to drop and which ones to keep, is started

In [ ]:
df.columns

Out of the columns of the dataset, the ones considered important were: 

- time
- year
- age
- phase
- diameter
- distance

Out of these, the column time was grouped by the values on 00:00:00.

In [24]:
df["time"] = pd.to_datetime(df["time"], errors="coerce")

df["time"] = df["time"].dt.normalize() 

daily = (
    df.sort_values("time")          
      .groupby("time", as_index=False)
      .agg({
          "time": "first",
          "year": "first",
          "age": "first",
          "phase": "first",
          "diameter": "first",
          "distance": "first"
      })
)

daily.to_csv("moon_phases_clean.csv", index=False)

print(len(daily), "rows in grouped dataset")

print(daily.head())
print(daily.columns)



2192 rows in grouped dataset
        time  year     age  phase  diameter  distance
0 2020-01-01  2020   6.491  36.26    1771.7    404538
1 2020-01-02  2020   7.491  45.51    1772.6    404346
2 2020-01-03  2020   8.366  53.78    1778.0    403098
3 2020-01-04  2020   9.699  66.27    1794.7    399349
4 2020-01-05  2020  10.366  72.27    1806.5    396742
Index(['time', 'year', 'age', 'phase', 'diameter', 'distance'], dtype='object')


After the grouping, the cleaning of the dataset takes place. Thaankfully the NASA dataset is already very clean so a function ```moon_phase_from_age(df)``` was developed to be able to add a string relative to the moon phase on the ```phase_str``` column


In [34]:
def moon_phase_from_age(df):
    
    df.loc[
        (df["age"] >= 0) & (df["age"] < 1.845),
        "phase_str"] = "New Moon"

    df.loc[
        (df["age"] >= 1.845) & (df["age"] < 5.536),
        "phase_str"] = "Waxing Crescent"

    df.loc[
        (df["age"] >= 5.536) & (df["age"] < 9.228),
        "phase_str"] = "First Quarter"

    df.loc[
        (df["age"] >= 9.228) & (df["age"] < 12.919), 
        "phase_str"] = "Waxing Gibbous"

    df.loc[
        (df["age"] >= 12.919) & (df["age"] < 16.610),
        "phase_str"] = "Full Moon"

    df.loc[
        (df["age"] >= 16.610) & (df["age"] < 20.302),
        "phase_str"] = "Waning Gibbous"

    df.loc[
        (df["age"] >= 20.302) & (df["age"] < 23.993), 
        "phase_str"] = "Last Quarter"

    df.loc[
        (df["age"] >= 23.993) & (df["age"] <= 29.53), 
        "phase_str"] = "Waning Crescent"
    
    return df


After this the function were applied to the dataset, cleaning it, and final manually inspected

In [38]:
df = moon_phase_from_age(df)

df["time"] = pd.to_datetime(df["time"], errors="coerce")

df["time"] = df["time"].dt.normalize() 

daily = (
    df.sort_values("time")          
      .groupby("time", as_index=False)
      .agg({
          "time": "first",
          "year": "first",
          "age": "first",
          "phase": "first",
          "diameter": "first",
          "distance": "first",
          "phase_str": "first"
      })
)

daily.to_csv("moon_phases_clean.csv", index=False)

print(len(daily), "rows in grouped dataset")

print(daily.head())
print(daily.columns)

2192 rows in grouped dataset
        time  year     age  phase  diameter  distance       phase_str
0 2020-01-01  2020   6.491  36.26    1771.7    404538   First Quarter
1 2020-01-02  2020   7.491  45.51    1772.6    404346   First Quarter
2 2020-01-03  2020   8.366  53.78    1778.0    403098   First Quarter
3 2020-01-04  2020   9.699  66.27    1794.7    399349  Waxing Gibbous
4 2020-01-05  2020  10.366  72.27    1806.5    396742  Waxing Gibbous
Index(['time', 'year', 'age', 'phase', 'diameter', 'distance', 'phase_str'], dtype='object')


In [39]:
daily.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2192 entries, 0 to 2191
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   time       2192 non-null   datetime64[ns]
 1   year       2192 non-null   int64         
 2   age        2192 non-null   float64       
 3   phase      2192 non-null   float64       
 4   diameter   2192 non-null   float64       
 5   distance   2192 non-null   int64         
 6   phase_str  2192 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(1)
memory usage: 252.6 KB


To end the Data wragling process, the clean dataset is moved to the Postgrees

In [ ]:
import sqlalchemy
from dotenv import load_dotenv
import os

load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
    f"@{os.environ.get('PGHOST', 'localhost')}:{os.environ.get('PGPORT', '5432')}/"
    f"{os.environ['PGDATABASE']}"
)
engine = sqlalchemy.create_engine(pg_url, pool_pre_ping=True)

raw_df = daily.copy()

with engine.begin() as conn:
    conn.execute(sqlalchemy.text("""
        CREATE TABLE IF NOT EXISTS clean_moon_phases (
            id BIGSERIAL PRIMARY KEY,
            time DATE,
            phase_str TEXT,
            distance INTEGER,
            year INTEGER,
            age FLOAT,
            phase FLOAT,
            diameter FLOAT
        )
    """))

raw_df.to_sql(
    "clean_moon_phases",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
    dtype={
        "time": sqlalchemy.Date(),
        "phase_str": sqlalchemy.Text(),
        "distance": sqlalchemy.Integer(),
        "year": sqlalchemy.Integer(),
        "age": sqlalchemy.Float(),
        "phase": sqlalchemy.Float(),
        "diameter": sqlalchemy.Float()
    },
)
print(f"Inserted {len(raw_df)} rows into Postgres")


Inserted 2192 raw rows into Postgres
